IMPORTS DE LAS LIBRERÍAS NECESARIAS

In [1]:
from ultralytics import YOLO # Modelo YOLO y entrenar nuestro detector de matrículas
import cv2 # Procesamiento de imágenes
from collections import defaultdict # Manejar diccionarios con listas
import csv # Manejar archivos CSV
from PIL import Image # Manejar imágenes
import torch # Manejo de tensores y modelos
from transformers import AutoProcessor, AutoModelForVision2Seq # Modelos de visión a secuencia
import numpy as np  # Manejo de arreglos numéricos
import math # Funciones matemáticas
import easyocr # Reconocimiento óptico de caracteres (OCR)
import time # Medir tiempos de ejecución
import pandas as pd # Manejo de datos en estructuras DataFrame
import matplotlib.pyplot as plt # Visualización de datos


c:\Users\mario\anaconda3\envs\VC_P4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INTENTO DE DETECCIÓN DE MATRÍCULAS BASADA EN CONTORNOS

In [3]:

# Carga del modelo
model = YOLO('yolo11n.pt')

filename = "C0142.mp4"
cap = cv2.VideoCapture(filename)

cv2.namedWindow('Deteccion con YOLO', cv2.WINDOW_NORMAL)
cv2.resizeWindow('Deteccion con YOLO', 1280, 720)

detections_count = 0

# funcion para detectar matrículas
def detect_plate(car_region):
    # paso la imagen a escala de grises
    gris = cv2.cvtColor(car_region, cv2.COLOR_BGR2GRAY)

    # suavizo para reducir el ruido
    gris = cv2.GaussianBlur(gris, (5, 5), 0)

    # aplico umbral adaptativo para destacar los bordes
    img_th1 = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 5) 

    # obtengo los contornos externos
    contornos, _ = cv2.findContours(img_th1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # comprobamos los contornos
    candidatos_mat = []
    for contorno in contornos:
        # aproximo el contorno a un polígono
        perimeter = cv2.arcLength(contorno, True)
        aprox = cv2.approxPolyDP(contorno, 0.018 * perimeter, True)

        # obtengo el rectángulo
        x, y, w, h = cv2.boundingRect(aprox)

        # características para identificar la matrícula
        aspect_ratio = w / float(h)
        area = w * h
        car_area = car_region.shape[0] * car_region.shape[1]
        relative_area = area / car_area

        if (2.0 <= aspect_ratio <= 5.5 and 
            0.01 <= relative_area <= 0.15 and
            w > 40 and h > 10):

            # Puntuación basada en qué tan cerca está del ratio ideal
            ideal_ratio = 4.5
            ratio_score = 1 - abs(aspect_ratio - ideal_ratio) / ideal_ratio
            
            candidatos_mat.append({
                'contour': aprox,
                'bbox': (x, y, w, h),
                'score': ratio_score * relative_area,
                'aspect_ratio': aspect_ratio
            })

    # cogemos el mejor candidato
    if candidatos_mat:
        mejor_matricula = max(candidatos_mat, key=lambda x: x['score'])
        return mejor_matricula

while cap.isOpened():
    ret, frame = cap.read()

    # si no hay imagen salimos
    if not ret:
        break

    # se ejecuta el modelo en el frame y se añaden los recuadros
    results = model(frame, classes=[2, 3, 5, 7], conf=0.5, verbose=False)
    annotated_frame = results[0].plot()

    # busco las matrículas de cada vehículo
    for result in results[0].boxes.data:
        x1, y1, x2, y2, conf, cls = result
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        
        # Extraer región del vehículo
        car_region = frame[y1:y2, x1:x2].copy()

        if car_region.size > 0:
            # busco la matricula
            plate = detect_plate(car_region)
            
            # si la encuentro, la dibujo
            if plate is not None:
                detections_count += 1
                px, py, pw, ph = plate['bbox']
                
                # Ajustar coordenadas al frame completo
                px += x1
                py += y1
                
                # Dibujar rectángulo de la matrícula en rojo
                cv2.rectangle(annotated_frame, (px, py), (px + pw, py + ph), (0, 0, 255), 2)
                cv2.putText(annotated_frame, 'PLATE', (px, py - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    
    frame = annotated_frame

    cv2.imshow('Deteccion con YOLO', annotated_frame)

    # se sale con ESC o Q/q
    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q') or key == ord('Q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"\nProcesamiento finalizado. Total de matrículas detectadas: {detections_count}")


Procesamiento finalizado. Total de matrículas detectadas: 119


CELDA DE ENTRENAMIENTO MODELO MATRÍCULAS

In [ ]:
# Descomentar para entrenar de nuevo el modelo YOLO en las matrículas

"""
from ultralytics import YOLO

# Cargar modelo preentrenado
model = YOLO("yolo11n.pt")

# Entrenar
model.train(
    data="dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo_matriculas",
    device=0
)

# Predicción en validación
results = model.predict(source='C:/Users/juanf/Desktop/Large-License-Plate-Detection-Dataset/images/val', save=True)
"""

FUNCIÓN PARA EXTRAER TEXTO DE MATRÍCULA CON smolVLM

In [6]:

# Establecemos el dispositivo para el modelo
device = "cuda" if torch.cuda.is_available() else "cpu"

# Cargamos el modelo SmolVLM-Instruct que es adecuado para tareas de visión a secuencia
model_name = "HuggingFaceTB/SmolVLM-Instruct"

# Cargamos el procesador asociado al modelo
processor = AutoProcessor.from_pretrained(model_name)

# Cargamos el modelo con el tipo de dato adecuado según el dispositivo
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Función para extraer texto de matrícula usando SmolVLM
def extraer_texto_matricula_smolVLM(frame, x1, y1, x2, y2):
    """
    Extrae el texto de una matrícula usando SmolVLM
    """

    try:
        # Extraemos la región de interés (ROI) de la matrícula con un pequeño padding
        padding = 5
        roi = frame[max(0, y1-padding):min(frame.shape[0], y2+padding), 
                   max(0, x1-padding):min(frame.shape[1], x2+padding)]
        
        # Verificamos si la ROI está vacía
        if roi.size == 0:
            return ""

        # Convertimos de BGR (OpenCV) a RGB (PIL)
        roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(roi_rgb)
        
        # Prompt específico para lectura de matrículas
        prompt = "Read the license plate number in this image. Only output the alphanumeric characters you see, without spaces or additional text."
        
        # Preparamos la plantilla de entrada para el modelo (imagen + prompt)
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        # Aplicamos la plantilla
        text = processor.apply_chat_template(messages, add_generation_prompt=True)

        # Cargamos la imagen y el texto al procesador
        inputs = processor(text=[text], images=[pil_image], return_tensors="pt")

        # Movemos los tensores al dispositivo adecuado
        inputs = inputs.to(device)
        
        # Generamos la predicción sin calcular gradientes(no es entrenamiento)
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs, # Desempaquetamos los inputs
                max_new_tokens=15, # Máximo número de tokens a generar
                do_sample=False 
            )

        # Decodificamos resultado
        generated_texts = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # Extraemos solo el texto de la respuesta
        texto = generated_texts[0].split("Assistant:")[-1].strip()

        # Limpiamos el texto: solo alfanuméricos
        texto_limpio = ''.join(c for c in texto if c.isalnum()).upper()
        
        return texto_limpio
    
    except Exception as e:
        print(f"Error al procesar matrícula con SmolVLM: {e}")
        return ""

FUNCIÓN PARA EXTRAER TEXTO DE MATRÍCULA CON EASYOCR

In [7]:
# Inicializamos el lector EasyOCR para español e inglés
reader = easyocr.Reader(['es', 'en'], gpu=torch.cuda.is_available())

def extraer_texto_matricula_easyOCR(frame, x1, y1, x2, y2):
    """
    Extrae el texto de una matrícula usando easyOCR - VERSION MEJORADA
    """
    try:
        # Extraer ROI de la matrícula con un pequeño padding
        padding = 5
        roi = frame[max(0, y1-padding):min(frame.shape[0], y2+padding), 
                   max(0, x1-padding):min(frame.shape[1], x2+padding)]
        
        if roi.size == 0:
            return ""
        
        # MEJORA 1: Redimensionar imagen si es muy pequeña
        height, width = roi.shape[:2]
        if height < 50 or width < 150:
            scale_factor = max(50/height, 150/width)
            new_width = int(width * scale_factor)
            new_height = int(height * scale_factor)
            roi = cv2.resize(roi, (new_width, new_height), interpolation=cv2.INTER_CUBIC)
        
        # MEJORA 2: Probar con la imagen original
        resultados_original = reader.readtext(
            roi,
            allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ',
            paragraph=False,
            detail=1,
            batch_size=1
        )
        
        # MEJORA 3: Probar también con preprocesamiento
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(gray)
        _, thresh = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        resultados_procesada = reader.readtext(
            thresh,
            allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ',
            paragraph=False,
            detail=1,
            batch_size=1
        )
        
        # Combinar resultados y elegir el mejor
        todos_resultados = resultados_original + resultados_procesada
        
        if not todos_resultados:
            return ""
        
        # Filtrar resultados con confianza muy baja
        resultados_filtrados = [r for r in todos_resultados if r[2] > 0.3]
        
        if not resultados_filtrados:
            return ""
        
        # Elegir el resultado con mayor confianza
        mejor_resultado = max(resultados_filtrados, key=lambda x: x[2])
        text = mejor_resultado[1]
        
        # Limpiar el texto: solo alfanuméricos
        texto_limpio = ''.join(c for c in text if c.isalnum()).upper()
        
        # MEJORA 4: Filtrar resultados que son demasiado cortos o largos
        if len(texto_limpio) < 4 or len(texto_limpio) > 12:
            return ""
        
        return texto_limpio
    
    except Exception as e:
        print(f"Error al procesar matrícula con EasyOCR: {e}")
        return ""
    

Using CPU. Note: This module is much faster with a GPU.


# PROCESAMIENTO DEL VIDEO

CONFIGURACIÓN DE PARÁMETROS

In [8]:
# Cargamos el modelo general de detección de objetos YOLOv11
general = YOLO("yolo11n.pt")

# Cargamos el modelo específico entrenado para detección de matrículas
matriculas = YOLO("runs/detect/yolo_matriculas/weights/best.pt")

# Declaramos el path del video a procesar
video_path = "C0142.MP4"

# Declaramos el path del tracker
tracker = "bytetrack.yaml"

# Declaramos las confianzas 
conf_general = 0.5
conf_plate = 0.3

# Definimos las clases de interés para la detección general
classes_general = [0, 2, 3, 5, 7]  # person, car, motorcycle, bus, truck

# Procesar OCR solo cada N frames
OCR_CADA_N_FRAMES = 5

FUNCIÓN DE PROCESAMIENTO

In [ ]:

def procesar_video(modelo_ocr: str):
    """
    Procesa un video detectando vehículos y extrayendo matrículas.

    Parámetro:
        modelo_ocr (str): 'smolVLM' o 'easyOCR'

    Retorna:
        dict: resumen estadístico del procesamiento.
    """

    # Validemos si el modelo OCR es válido
    if modelo_ocr not in ["smolVLM", "easyOCR"]:
        raise ValueError("Modelo OCR no válido. Usa 'smolVLM' o 'easyOCR'.")

    # Declaramos el nombre de los archivos de salida en función del modelo OCR
    nombre_base = f"resultado_{modelo_ocr}"
    output_video = f"{nombre_base}.mp4"
    output_csv = f"{nombre_base}.csv"

    # Iniciamos variables de almacenamiento
    mejores_matriculas_por_vehiculo = {}
    info_vehiculos = {}

    # Tiempo de inicio
    start_time = time.time()

    # Procesamos el video con tracking (modo stream porque es mejor para videos largos)
    results_stream = general.track(
        source=video_path,
        tracker=tracker,
        classes=classes_general,
        conf=conf_general,
        persist=True,
        stream=True
    )

    # Variables para escritura de video
    h, w, fps = None, None, 30
    writer = None
    frame_count = 0

    # Vamos procesando frame a frame del resultado del tracking
    for frame_num, r in enumerate(results_stream):

        # Actualizamos el conteo de frames procesados
        frame_count += 1

        # Copiamos el frame original y el frame con tracking
        frame = r.orig_img.copy()
        tracked_frame = r.plot()

        # Inicializamos el escritor de video si es la primera vez
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

        # Declaramos una lista de objetos trackeados en este frame
        objetos_trackeados = []

        # Si el frame tiene cajas detectadas las procesamos
        if hasattr(r, "boxes") and r.boxes is not None:
            for box in r.boxes:
                if box.id is None:
                    continue

                # Tomamos la clase, ID, confianza y coordenadas de la caja
                cls = int(box.cls)
                track_id = int(box.id)
                conf = float(box.conf)
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                tipo = general.names[cls]

                # Si es la primera vez que vemos este vehículo, lo registramos
                if track_id not in info_vehiculos:
                    info_vehiculos[track_id] = {
                        'tipo': tipo,
                        'conf': conf,
                        'primer_frame': frame_num,
                        'ultimo_frame': frame_num,
                        'bbox': (x1, y1, x2, y2)
                    }
                # Si ya lo habíamos visto, actualizamos la info
                else:
                    info_vehiculos[track_id]['ultimo_frame'] = frame_num
                    info_vehiculos[track_id]['bbox'] = (x1, y1, x2, y2)

                # Añadimos a la lista de objetos trackeados en este frame
                objetos_trackeados.append({'track_id': track_id, 'tipo': tipo, 'bbox': (x1, y1, x2, y2)})

        # Si es el frame adecuado, realizamos OCR de matrículas
        if frame_num % OCR_CADA_N_FRAMES == 0:

            # Realizamos detección de matrículas en el frame completo
            res_plate = matriculas(frame, conf=conf_plate, verbose=False)[0]

            # Procesamos cada caja de matrícula detectada
            if hasattr(res_plate, "boxes") and res_plate.boxes is not None:
                for pbox in res_plate.boxes:

                    # Obtenemos las coordenadas y confianza de la caja de matrícula
                    x1m, y1m, x2m, y2m = map(int, pbox.xyxy[0].tolist())
                    conf_plate_det = float(pbox.conf)

                    # Extraemos el texto de la matrícula usando el modelo seleccionado
                    if modelo_ocr == "smolVLM":
                        texto_matricula = extraer_texto_matricula_smolVLM(frame, x1m, y1m, x2m, y2m)
                    else:
                        texto_matricula = extraer_texto_matricula_easyOCR(frame, x1m, y1m, x2m, y2m)

                    if not texto_matricula:
                        continue

                    # Asociamos la matrícula a vehículo correspondiente
                    for obj in objetos_trackeados:

                        # Filtramos lo que no son vehículos
                        if obj['tipo'] not in ['car', 'motorcycle', 'bus', 'truck']:
                            continue

                        # Cogemos las coordenadas del vehículo
                        x1v, y1v, x2v, y2v = obj['bbox']
                        track_id = obj['track_id']

                        # Calculamos área de intersección entre vehículo y matrícula
                        intersecta_x = max(0, min(x2v, x2m) - max(x1v, x1m))
                        intersecta_y = max(0, min(y2v, y2m) - max(y1v, y1m))
                        area_interseccion = intersecta_x * intersecta_y

                        # Si hay intersección, asociamos la matrícula al vehículo 
                        if area_interseccion > 0:
                            score = conf_plate_det * (len(texto_matricula) / 10.0)

                            # Si es la mejor matrícula para este vehículo, la guardamos
                            if track_id not in mejores_matriculas_por_vehiculo or score > mejores_matriculas_por_vehiculo[track_id]['score']:
                                mejores_matriculas_por_vehiculo[track_id] = {
                                    'frame': frame_num,
                                    'bbox_vehiculo': (x1v, y1v, x2v, y2v),
                                    'bbox_matricula': (x1m, y1m, x2m, y2m),
                                    'texto': texto_matricula,
                                    'conf_mat': conf_plate_det,
                                    'score': score,
                                    'tipo': obj['tipo'],
                                    'conf_obj': info_vehiculos[track_id]['conf']
                                }
                            break

        # Creamos el frame anotado para el video de salida
        annotated = tracked_frame.copy()

        # Recorremos los objetos trackeados para dibujarles la info de matrícula
        for obj in objetos_trackeados:

            # # Si el objeto es una persona la difuminamos por privacidad
            # if obj['tipo'] == 'person':
            #     x1p, y1p, x2p, y2p = obj['bbox']
            
            #     x1p, y1p = max(0, x1p), max(0, y1p)
            #     x2p, y2p = min(w, x2p), min(h, y2p)
            #     if x2p > x1p and y2p > y1p:
            #         annotated[y1p:y2p, x1p:x2p] = cv2.GaussianBlur(
            #             annotated[y1p:y2p, x1p:x2p],
            #             (51, 51), 0
            #         )

            track_id = obj['track_id']

            if track_id in mejores_matriculas_por_vehiculo:
                x1v, y1v, x2v, y2v = obj['bbox']
                texto_mat = mejores_matriculas_por_vehiculo[track_id]['texto']
                texto = f"ID:{track_id} - {texto_mat}"

                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.7
                thickness = 2
                (text_width, text_height), baseline = cv2.getTextSize(texto, font, font_scale, thickness)

                text_x = x1v
                text_y = y2v + text_height + 10

                cv2.rectangle(
                    annotated,
                    (text_x, text_y - text_height - 5),
                    (text_x + text_width + 10, text_y + 5),
                    (0, 255, 0),
                    -1
                )
                cv2.putText(annotated, texto, (text_x + 5, text_y),
                            font, font_scale, (255, 255, 255), thickness)

            # # PARA DIFUMINAR LAS MATRICULAS, lo hacemos justo antes de guardar el frame para que los ocr puedan leerlas
            # for track_id, info in mejores_matriculas_por_vehiculo.items():
            #     x1m, y1m, x2m, y2m = map(int, info['bbox_matricula'])
            #     # recortamos coordenadas al tamaño del frame
            #     x1m, y1m = max(0, x1m), max(0, y1m)
            #     x2m, y2m = min(w, x2m), min(h, y2m)
            #     # aplicamos solo si la caja tiene área válida
            #     if x2m > x1m and y2m > y1m:
            #         annotated[y1m:y2m, x1m:x2m] = cv2.GaussianBlur(
            #             annotated[y1m:y2m, x1m:x2m],
            #             (51, 51), 0
            #         )


        writer.write(annotated)

    # Cerramos el escritor de video
    if writer is not None:
        writer.release()

    # Creamos el CSV resumen
    vehiculos_con_matricula = 0
    conteo_clases = {}
    vehiculos_salen_derecha = 0
    vehiculos_salen_izquierda = 0
    personas_salen_derecha = 0
    personas_salen_izquierda = 0

    with open(output_csv, 'w', newline='', encoding='utf-8') as f_csv:
        csv_writer = csv.writer(f_csv)
        csv_writer.writerow([
            'track_id', 'tipo_vehiculo', 'conf_vehiculo',
            'frame_primera_aparicion', 'frame_ultima_aparicion',
            'matricula_detectada', 'texto_matricula',
            'conf_matricula', 'frame_mejor_deteccion',
            'vehiculo_x1', 'vehiculo_y1', 'vehiculo_x2', 'vehiculo_y2',
            'matricula_x1', 'matricula_y1', 'matricula_x2', 'matricula_y2'
        ])

        # Escribimos la información individual de cada objeto
        for track_id, info in info_vehiculos.items():
            tipo = info['tipo']
            x1, y1, x2, y2 = info['bbox']

            # Conteo de clases
            conteo_clases[tipo] = conteo_clases.get(tipo, 0) + 1

            # Verificamos si el objeto sale por los lados
            if tipo in ['car', 'motorcycle', 'bus', 'truck']:
                if x1 <= 5:
                    vehiculos_salen_izquierda += 1
                elif x2 >= w - 5:
                    vehiculos_salen_derecha += 1
            elif tipo == 'person':
                if x1 <= 5:
                    personas_salen_izquierda += 1
                elif x2 >= w - 5:
                    personas_salen_derecha += 1

            # Escribimos los datos del objeto en el CSV
            if track_id in mejores_matriculas_por_vehiculo:
                mat = mejores_matriculas_por_vehiculo[track_id]
                vehiculos_con_matricula += 1
                csv_writer.writerow([
                    track_id, mat['tipo'], f"{mat['conf_obj']:.2f}",
                    info['primer_frame'], info['ultimo_frame'], 'Si',
                    mat['texto'], f"{mat['conf_mat']:.2f}", mat['frame'],
                    *mat['bbox_vehiculo'], *mat['bbox_matricula']
                ])
            else:
                csv_writer.writerow([
                    track_id, info['tipo'], f"{info['conf']:.2f}",
                    info['primer_frame'], info['ultimo_frame'], 'No',
                    '', '', '', *info['bbox'], '', '', '', ''
                ])

        # Añadimos líneas en blanco y resumen al final del CSV
        csv_writer.writerow([])
        csv_writer.writerow(['=== RESUMEN ESTADÍSTICO ==='])
        csv_writer.writerow(['Conteo por clase:'])
        for clase, count in conteo_clases.items():
            csv_writer.writerow([clase, count])

        csv_writer.writerow([])
        csv_writer.writerow(['Vehículos que salen por la izquierda', vehiculos_salen_izquierda])
        csv_writer.writerow(['Vehículos que salen por la derecha', vehiculos_salen_derecha])
        csv_writer.writerow(['Personas que salen por la izquierda', personas_salen_izquierda])
        csv_writer.writerow(['Personas que salen por la derecha', personas_salen_derecha])

    # Calculamos tiempos y estadísticas finales para futura comparación
    end_time = time.time()
    tiempo_total = end_time - start_time
    total_frames = frame_count
    total_vehiculos = len(info_vehiculos)
    total_con_matricula = vehiculos_con_matricula

    # Devolvemos el resumen estadístico
    return {
        "modelo": modelo_ocr,
        "video_salida": output_video,
        "csv_salida": output_csv,
        "tiempo_total": tiempo_total,
        "frames": total_frames,
        "tiempo_medio_frame": tiempo_total / total_frames if total_frames else 0,
        "vehiculos_detectados": total_vehiculos,
        "matriculas_detectadas": total_con_matricula,
        "tasa_acierto": total_con_matricula / total_vehiculos if total_vehiculos else 0
    }


# ANÁLISIS Y COMPARATIVA (SmolVLM vs EasyOCR)

In [11]:

# Procesamos el video con ambos modelos OCR
result_smol = procesar_video("smolVLM")
result_easy = procesar_video("easyOCR")

# Creamos DataFrame para comparación
comparativa = pd.DataFrame([
    {
        "Modelo": result_smol["modelo"],
        "Tiempo total (s)": result_smol["tiempo_total"],
        "Tiempo medio por frame (s)": result_smol["tiempo_medio_frame"],
        "Tasa de acierto (%)": result_smol["tasa_acierto"] * 100
    },
    {
        "Modelo": result_easy["modelo"],
        "Tiempo total (s)": result_easy["tiempo_total"],
        "Tiempo medio por frame (s)": result_easy["tiempo_medio_frame"],
        "Tasa de acierto (%)": result_easy["tasa_acierto"] * 100
    }
])

# Graficamos los resultados para el tiempo total de inferencia
plt.figure(figsize=(8, 5))
plt.bar(comparativa["Modelo"], comparativa["Tiempo total (s)"])
plt.title("Comparativa de tiempo total de inferencia")
plt.ylabel("Tiempo total (s)")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

# Graficamos los resultados para la tasa de acierto
plt.figure(figsize=(8, 5))
plt.bar(comparativa["Modelo"], comparativa["Tasa de acierto (%)"], color="green")
plt.title("Comparativa de tasa de acierto en detección de matrículas")
plt.ylabel("Tasa de acierto (%)")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

# Mostramos la tabla comparativa
comparativa



video 1/1 (frame 1/2832) c:\UNIVERSIDAD\Ao_4\1Semestre\VC\P1\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 160.3ms
video 1/1 (frame 2/2832) c:\UNIVERSIDAD\Ao_4\1Semestre\VC\P1\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 54.4ms
video 1/1 (frame 3/2832) c:\UNIVERSIDAD\Ao_4\1Semestre\VC\P1\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 62.7ms
video 1/1 (frame 4/2832) c:\UNIVERSIDAD\Ao_4\1Semestre\VC\P1\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 49.6ms
video 1/1 (frame 5/2832) c:\UNIVERSIDAD\Ao_4\1Semestre\VC\P1\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 52.5ms
video 1/1 (frame 6/2832) c:\UNIVERSIDAD\Ao_4\1Semestre\VC\P1\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 55.6ms
video 1/1 (frame 7/2832) c:\UNIVERSIDAD\Ao_4\1Semestre\VC\P1\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 49.8ms
video 1/1 (frame 8/2832) c:\UNIVERSIDAD\Ao_4\1Semestre\VC\P1\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 50.5ms
video 1/1 (frame 9/2832) c:\UNIVERSIDAD\Ao_4\1Semestre\VC\P1\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 49.2ms
video 1/

KeyboardInterrupt: 